📊 EDA Tabular - Objetivos:

A) UNIVARIADO
   ├─ Distribución por variable (histogramas, boxplots)
   ├─ Sparsity: % de ceros por variable
   ├─ Variables constantes (varianza = 0)
   ├─ Variables casi-constantes (p.ej., varianza muy baja)
   └─ Rango de valores (min, max, mean, std)

B) BIVARIADO
   ├─ Correlación de Pearson (matriz completa)
   ├─ Identificar clusters de correlación alta (>0.95)
   ├─ Variables redundantes a eliminar
   └─ Visualizar top 25 correlaciones

C) POR CLASE
   ├─ Media/mediana de características por clase
   ├─ Características más discriminativas (diferencia entre clases)
   ├─ Variables específicas de cada familia de malware
   └─ Tabla: top 10 features por clase

D) SEPARABILIDAD
   ├─ PCA (2D/3D) - visualizar clusters
   ├─ t-SNE - comparar con estudio previo
   ├─ UMAP - alternativa moderna
   └─ Ratio varianza explicada (PCA)

E) DETECCIÓN DE ANOMALÍAS
   ├─ Isolation Forest para outliers
   ├─ Muestras extremas (distancia Mahalanobis)
   ├─ Posibles datos corruptos
   └─ Contar/visualizar outliers por clase

🔧 Pipeline de Limpieza y Transformación:

PASO 1: EXTRACCIÓN Y CODIFICACIÓN
  • Separar apk_name (guardar índice para traceabilidad)
  • Codificar Class: LabelEncoder (0-4) o dict
  • Crear split X, y

PASO 2: LIMPIEZA DE CARACTERÍSTICAS
  • Eliminar variables constantes (var = 0)
  • Eliminar variables quasi-constantes (var < 0.01)
  • Eliminar colinealidad extrema (corr > 0.99)
    → Conservar variable con mayor varianza
  • Feature selection option:
    - Univariada: SelectKBest, chi2 (para count data)
    - Mutual Information
    - Eliminar features con <1% variación

PASO 3: MANEJO DE DESBALANCE
  ⚠️ CRÍTICO por diseño del dataset
  
  Opciones evaluadas:
  a) class_weight='balanced' (Sklearn)
     ✓ Simple, no manipula datos
     ✓ Recomendado primero
  
  b) SMOTE + TomekLinks (imblearn)
     ✓ Oversampling minoritarias + undersampling mayoritarias
     ✓ Mejor si train es pequeño (no es caso)
  
  c) Random stratified undersampling
     ✓ Si memoria es issue
  
  d) Threshold optimization post-entrenamiento
     ✓ Mover decision boundary para Adware

  RECOMENDACIÓN: Empezar con class_weight='balanced'
                 Si macro-F1 sigue bajo, aplicar SMOTE

PASO 4: ESCALADO
  • StandardScaler vs MinMaxScaler vs RobustScaler
    - StandardScaler: Default, asume normalidad
    - RobustScaler: Si hay outliers (más robusto)
  • Aplicar SOLO a train, transform test/val
  • Importante para: LogReg, SVM, KNN, MLP
  • NO necesario para: Tree-based, XGBoost

PASO 5: EVITAR DATA LEAKAGE
  • Fit scaler en train split únicamente
  • Fit selección características en train
  • Fit técnicas de balanceo EN FOLD INTERNO (cross-val)
  • Test/validation nunca toca pipeline

📐 Estrategia de Split Reproducible:

DISEÑO PROPUESTO:
│
├─ TRAIN (70% = 8,178)
│  ├─ [Entrenar modelos, hyperparameter tuning]
│  └─ Dentro: 5-fold stratified CV para validación
│
├─ VALIDATION (15% = 2,505)
│  └─ [Seleccionar mejor modelo, ajustar threshold]
│
└─ TEST (15% = 2,505) 
   └─ [FINAL: Reporte oficial, sin tocar]

CÓDIGO ESTRUCTURA:
```python
from sklearn.model_selection import train_test_split

# Primer split: Separar test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)

# Segundo split: Separar train (70%) y val (15%)  
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176,  # ~15% of total
    stratify=y_temp, random_state=42
)

# Cross-validation estratificada (DENTRO de train)
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

🤖 Arquitectura de Modelos - Stack Recomendado:

NIVEL 1: BASELINE (SIEMPRE)
  ├─ DummyClassifier (estrategia='most_frequent')
  │  └─ Accuracy esperado: ~29% (predice SMS siempre)
  │     Macro-F1 esperado: ~20%
  │  └─ PROPÓSITO: Validar que otros modelos mejoren
  │
  └─ Logistic Regression
     ├─ Ventajas: Interpretable, rápido, baseline lineal
     ├─ Hiperparámetros: C, solver, class_weight
     └─ Métrica esperada: Macro-F1 ~70-75%

NIVEL 2: MODELOS CLÁSICOS POTENTES
  ├─ Random Forest
  │  ├─ Ventajas: No necesita scaling, feature importance, robusto
  │  ├─ Hiperparámetros: n_estimators, max_depth, min_samples_leaf
  │  ├─ class_weight: 'balanced_subsample'
  │  └─ Métrica esperada: Macro-F1 ~80-85%
  │
  ├─ SVM (LinearSVC con escalado)
  │  ├─ Ventajas: Kernel lineal rápido, escalable
  │  ├─ Hiperparámetros: C, class_weight
  │  ├─ REQUIERE: RobustScaler (400 features)
  │  └─ Métrica esperada: Macro-F1 ~75-80%
  │
  └─ MLP Tabular Simple
     ├─ Ventajas: No-lineal, aprende patrones complejos
     ├─ Arquitectura: 400 → 256 → 128 → 64 → 5
     ├─ Regularización: L2, Dropout, Early Stopping
     ├─ REQUIERE: StandardScaler
     └─ Métrica esperada: Macro-F1 ~75-82%

NIVEL 3: GRADIENT BOOSTING (SI COMPUTA)
  ├─ XGBoost
  │  ├─ Ventajas: SOTA para tabulares, feature importance, rápido
  │  ├─ Hiperparámetros: max_depth, learning_rate, subsample, colsample
  │  ├─ scale_pos_weight para desbalance
  │  └─ Métrica esperada: Macro-F1 ~85-90%
  │
  ├─ LightGBM (ligero, recomendado)
  │  ├─ Ventajas: MÁS RÁPIDO que XGBoost, memoria eficiente
  │  ├─ Hiperparámetros: num_leaves, learning_rate, feature_fraction
  │  └─ Métrica esperada: Macro-F1 ~85-90%
  │
  └─ CatBoost
     ├─ Ventajas: Maneja categorías nativamente, robusto
     ├─ Desventaja: Lento en train (pero predicción rápida)
     └─ Métrica esperada: Macro-F1 ~85-90%

RECOMENDACIÓN DE PRIORIDAD:
1. Logistic Regression (baseline)
2. Random Forest (robustez, interpretabilidad)
3. LightGBM (mejor performance, rápido)
4. XGBoost (si tiempo disponible)
5. SVM (si hipótesis lineal)

📈 Esquema de Evaluación - Énfasis en Desbalance:

MÉTRICAS OBLIGATORIAS:

a) MACRO-F1 (⭐ PRINCIPAL por desbalance)
   - Promedia F1 de cada clase sin peso
   - No castigada por clases mayoritarias
   - Rango: 0-1 (1 = perfecto)

b) MICRO-F1 (validación)
   - Promedia globalmente (penaliza minoritarias)
   - Debe estar >Macro-F1 si desbalance es problema

c) WEIGHTED-F1
   - Promedia ponderado por muestras
   - Entre Macro y Micro

d) ACCURACY
   - Recordar que NO es buena métrica aquí
   - Será ~70% solo prediendo mayoritarias

e) POR CLASE:
   - Precision, Recall, F1 para cada clase
   - CRÍTICO: Monitorear Adware por separado
   
   Ejemplo:
   ┌──────────┬───────────┬───────────┬──────────┐
   │ Clase    │ Precision │ Recall    │ F1-Score │
   ├──────────┼───────────┼───────────┼──────────┤
   │ Adware   │ 0.82      │ 0.75      │ 0.78     │ ← Vigilar
   │ Banking  │ 0.85      │ 0.81      │ 0.83     │
   │ Benign   │ 0.88      │ 0.90      │ 0.89     │
   │ Riskware │ 0.86      │ 0.84      │ 0.85     │
   │ SMS      │ 0.91      │ 0.88      │ 0.89     │
   ├──────────┼───────────┼───────────┼──────────┤
   │ MACRO    │ -         │ -         │ 0.847    │ ← Meta
   │ WEIGHTED │ -         │ -         │ 0.868    │
   └──────────┴───────────┴───────────┴──────────┘

f) MATRIZ DE CONFUSIÓN
   - Visualizar confusiones comunes
   - ¿Adware confundido con qué?
   - Heatmap anotado

g) CURVA ROC vs OVO/OVR (multiclass)
   - Si tiempo: One-vs-Rest AUC

UMBRAL DE ÉXITO:
  - Baseline (Dummy): Macro-F1 ~0.20
  - Modelo aceptable: Macro-F1 ≥ 0.80
  - Modelo bueno: Macro-F1 ≥ 0.85
  - SOTA (basado en literatura): Macro-F1 ≥ 0.90

🔍 Entendimiento de Decisiones del Modelo:

PARA TREE-BASED (Random Forest, XGBoost):
  ├─ Feature Importance (Gini/Split)
  │  ├─ Qué variables son más usadas
  │  ├─ Top 20 features
  │  └─ Visualizar barras
  │
  ├─ Permutation Importance
  │  ├─ Cuánto cae performance si scramblizamos feature
  │  ├─ Más interpretable que Gini
  │  └─ Calcular en TEST SET
  │
  └─ SHAP (SHapley Additive exPlanations)
     ├─ Explicación por muestra
     ├─ "Este APK es malware porque:"
     ├─ Dependencia de valores en cada feature
     └─ TreeExplainer para XGBoost (rápido)

PARA LINEAL (Logistic Regression, SVM):
  ├─ Coeficientes (weights)
  │  ├─ Positivos = empujan clase 1
  │  ├─ Negativos = empujan clase 0
  │  ├─ Magnitud = importancia
  │  └─ Top +/- 20 features por clase
  │
  └─ Ratio impacto: coef * X_mean

ANÁLISIS ESPECÍFICO MALWARE:
  ├─ Permisos más asociados a ADWARE:
  │  └─ p.ej., permission.SYSTEM_OVERLAY_WINDOW
  │
  ├─ Acciones críticas para BANKING:
  │  └─ p.ej., action.NOTIFICATION_ADD (credenciales)
  │
  ├─ Servicios sospechosos para SMS:
  │  └─ p.ej., SmsService, MessageService
  │
  ├─ Indicadores de RISKWARE:
  │  └─ Llamadas a funciones críticas (GetTasks, etc)
  │
  └─ Tabla resumen: TOP 3 features discriminativas por clase

ENTREGABLES:
  ├─ feature_importance_plot.png
  ├─ shap_summary.png
  └─ interpretation_report.txt